# Solving equations with SymPy: locating a source

This tutorial develops equation-solving practice through a source-localization problem. Distances from sensors constrain an unknown position, but those constraints can produce multiple candidates, a unique solution, or no exact solution. The important work is therefore not merely calling a solver: it is choosing the right mathematical problem, interpreting the complete solution set, and checking candidates against the original equations.

The exact measurements in the core example are synthetic and deliberately consistent. A later section perturbs one measurement to show why noisy experimental data should not usually be treated as a system of exact equalities. The notebook is intended for researchers with basic Python and algebra experience. Allow approximately 45–60 minutes for the core material and 10–15 minutes for the practice task.

Run the notebook in the repository's default Pixi environment; no notebook-local installation is required.

## Objectives

After working through the notebook, you should be able to:

- encode a mathematical equality with `Eq` and convert it to a residual expression;
- use a solution domain to express admissibility constraints;
- interpret all branches returned by a scalar or nonlinear-system solver;
- choose among `solveset`, `linsolve`, `nonlinsolve`, and `solve`;
- transform a small nonlinear localization system into a linear system without mistaking the transformed result for a verified solution;
- validate candidate solutions against the original equations;
- recognize when inconsistent measurements require numerical estimation rather than exact symbolic solving.

## Setup

Only SymPy is required. Exact integers, rational numbers, and square roots are retained throughout the synthetic example.

In [1]:
import sympy as sp

sp.init_printing()

## Range-measurement model

Let $(x,y)$ be the unknown source coordinates. A sensor at known coordinates $(a_i,b_i)$ reports a range $r_i$, where the subscript $i$ identifies the sensor. In the ideal model,

$$
(x-a_i)^2+(y-b_i)^2=r_i^2.
$$

Each equation describes a circle. The model assumes exact sensor positions, exact ranges, Euclidean geometry, and no systematic measurement error. Those assumptions are intentionally unrealistic for measured data and will be revisited later.

In [2]:
x, y = sp.symbols("x y", real=True)


def range_equation(sensor, measured_range):
    """Return the ideal range equation for one two-dimensional sensor."""
    sensor_x, sensor_y = sensor
    return sp.Eq(
        (x - sensor_x)**2 + (y - sensor_y)**2,
        measured_range**2,
    )

## One measurement can have multiple solutions

Begin in one dimension. A sensor at coordinate $s=2$ measures distance $d=3$ from a source at unknown coordinate $q$. The model is

$$
(q-s)^2=d^2.
$$

The geometry already suggests two possible positions, one on each side of the sensor.

In [3]:
q = sp.symbols("q", real=True)
sensor_position = 2
measured_distance = 3
position_equation = sp.Eq(
    (q - sensor_position)**2,
    measured_distance**2,
)

position_equation

### Equations and expressions

`Eq(left, right)` represents the equality $\text{left}=\text{right}$. Many algorithms instead work with a residual expression that should equal zero:

$$
F(q)=\text{left}-\text{right}.
$$

Here $F$ denotes the residual. Retaining both forms makes solver calls and later validation explicit.

In [4]:
position_residual = position_equation.lhs - position_equation.rhs
position_residual

### `solve` and `solveset`

`solve` is a mature general-purpose interface, but its return type depends on the problem and options. For one equation in one unknown, `solveset` has a more consistent set-based result and makes the solution domain explicit. Neither the order of a list nor the display order of a mathematical set gives one solution scientific priority over another.

In [5]:
solutions_from_solve = sp.solve(position_equation, q)
real_solution_set = sp.solveset(
    position_equation,
    q,
    domain=sp.S.Reals,
)

solutions_from_solve, real_solution_set

### Domains express prior constraints

Suppose the apparatus is known to occupy only nonnegative coordinates. That information belongs in the mathematical problem, rather than being used to discard an inconvenient answer after solving. The interval $[0,\infty)$ retains only the admissible solution.

In [6]:
nonnegative_solution_set = sp.solveset(
    position_equation,
    q,
    domain=sp.Interval(0, sp.oo),
)

nonnegative_solution_set

### Validate against the original equation

A candidate is accepted only if substitution into the original equation gives a zero residual. This check protects against transcription mistakes and against roots introduced by transformations such as squaring or clearing denominators.

In [7]:
def equation_residual(equation, values):
    """Evaluate and simplify left minus right for an equation."""
    return sp.simplify(
        (equation.lhs - equation.rhs).subs(values)
    )


{
    candidate: equation_residual(position_equation, {q: candidate})
    for candidate in real_solution_set
}

## Two sensors leave a geometric ambiguity

Return to two dimensions. Sensor A is at $(0,0)$ and sensor B at $(4,0)$. Both report the exact range $\sqrt{5}$. Their two range circles intersect at two points, reflected across the horizontal axis.

In [8]:
sensor_a = (0, 0)
sensor_b = (4, 0)
range_a = sp.sqrt(5)
range_b = sp.sqrt(5)

equation_a = range_equation(sensor_a, range_a)
equation_b = range_equation(sensor_b, range_b)

equation_a, equation_b

For a system, `solve(..., dict=True)` requests a list of mappings from unknowns to values. This avoids several other return formats supported by the general `solve` interface. Every mapping remains a candidate until scientific constraints distinguish it.

In [9]:
two_sensor_candidates = sp.solve(
    (equation_a, equation_b),
    (x, y),
    dict=True,
)

two_sensor_candidates

`nonlinsolve` is the specialized set-based interface for nonlinear systems. It accepts residual expressions and returns tuples in the same order as the supplied unknowns. For this finite polynomial system, it exposes the same two branches.

In [10]:
residual_a = equation_a.lhs - equation_a.rhs
residual_b = equation_b.lhs - equation_b.rhs

sp.nonlinsolve((residual_a, residual_b), (x, y))

Both candidates satisfy both original equations. Selecting the first list element would therefore resolve nothing; it would merely depend on output ordering.

In [11]:
[
    (
        candidate,
        equation_residual(equation_a, candidate),
        equation_residual(equation_b, candidate),
    )
    for candidate in two_sensor_candidates
]

## A third measurement resolves the ambiguity

Sensor C is at $(0,3)$ and reports range $\sqrt{8}$. Test the existing candidates against this new equation before solving the complete system again. This makes the role of the additional information explicit.

In [12]:
sensor_c = (0, 3)
range_c = sp.sqrt(8)
equation_c = range_equation(sensor_c, range_c)

[
    (candidate, equation_residual(equation_c, candidate))
    for candidate in two_sensor_candidates
]

In [13]:
admissible_candidates = [
    candidate
    for candidate in two_sensor_candidates
    if equation_residual(equation_c, candidate) == 0
]

admissible_candidates

## Exposing a linear system

Subtract the residual for sensor A from the residuals for sensors B and C. The $x^2$ and $y^2$ terms cancel, leaving two linear equations. These difference equations require the three original residuals to be equal; they do not by themselves require that common residual to be zero. Verification against the original circle equations therefore remains necessary.

In [14]:
residual_c = equation_c.lhs - equation_c.rhs
linearized_residuals = [
    sp.expand(residual_b - residual_a),
    sp.expand(residual_c - residual_a),
]

linearized_residuals

`linear_eq_to_matrix` extracts the matrix $A$ and right-hand-side vector $b$ from a system in the form $A z=b$, where $z=(x,y)^T$ is the vector of unknown coordinates. `linsolve` then returns a set of ordered solution tuples.

In [15]:
coefficient_matrix, right_hand_side = sp.linear_eq_to_matrix(
    linearized_residuals,
    (x, y),
)
linear_solution_set = sp.linsolve(
    (coefficient_matrix, right_hand_side),
    (x, y),
)

coefficient_matrix, right_hand_side, linear_solution_set

⎛⎡-8  0 ⎤  ⎡-16⎤          ⎞
⎜⎢      ⎥, ⎢   ⎥, {(2, 1)}⎟
⎝⎣0   -6⎦  ⎣-6 ⎦          ⎠

Only after confirming that the returned set is a singleton do we extract its sole tuple. Substitution into all three original equations verifies that the linearized result is a genuine solution in the consistent example.

In [16]:
assert len(linear_solution_set) == 1
linear_point = next(iter(linear_solution_set))
linear_candidate = dict(zip((x, y), linear_point, strict=True))
original_equations = (equation_a, equation_b, equation_c)
original_residuals = tuple(
    equation_residual(equation, linear_candidate)
    for equation in original_equations
)

linear_candidate, original_residuals

## Exact equations are a poor model for noisy measurements

Replace sensor C's exact range $\sqrt{8}\approx2.828$ by the nearby recorded value $2.85=57/20$. The rational representation records the stated decimal exactly; it does not claim that a physical measurement has infinite precision. The three ideal circles now have no common intersection.

In [17]:
noisy_range_c = sp.Rational(57, 20)
noisy_equation_c = range_equation(sensor_c, noisy_range_c)
noisy_residual_c = noisy_equation_c.lhs - noisy_equation_c.rhs

sp.nonlinsolve(
    (residual_a, residual_b, noisy_residual_c),
    (x, y),
)

The difference equations still form a solvable linear system. Its point equalizes the three squared-distance residuals, but does not make them zero. Calling that point “the solution” would silently change the problem.

In [18]:
noisy_linearized_residuals = [
    sp.expand(residual_b - residual_a),
    sp.expand(noisy_residual_c - residual_a),
]
noisy_matrix, noisy_right_hand_side = sp.linear_eq_to_matrix(
    noisy_linearized_residuals,
    (x, y),
)
noisy_linear_solution = sp.linsolve(
    (noisy_matrix, noisy_right_hand_side),
    (x, y),
)

noisy_linear_solution

In [19]:
assert len(noisy_linear_solution) == 1
noisy_point = next(iter(noisy_linear_solution))
noisy_candidate = dict(zip((x, y), noisy_point, strict=True))
noisy_original_equations = (
    equation_a,
    equation_b,
    noisy_equation_c,
)
noisy_original_residuals = tuple(
    sp.N(equation_residual(equation, noisy_candidate), 6)
    for equation in noisy_original_equations
)

noisy_candidate, noisy_original_residuals

For noisy measurements, a defensible next problem might minimize

$$
\sum_i w_i F_i(x,y)^2,
$$

where $F_i$ is the residual for sensor $i$ and $w_i$ is a weight derived from that sensor's uncertainty. This is an estimation problem, not the exact equation-solving problem addressed here. The choice of residual and weights changes the statistical meaning of the result, so an arbitrary numerical root or aggressively rounded input is not an adequate substitute.

## Choosing a solver

| Mathematical problem | Useful interface | Important limitation |
|---|---|---|
| One equation, one unknown, explicit domain | `solveset` | May return a conditional or unevaluated set |
| Linear system | `linsolve` | An underdetermined result may contain parameters |
| Small nonlinear system | `nonlinsolve` | Symbolic cost and expression size can grow rapidly |
| General convenience interface | `solve(..., dict=True)` | Return conventions otherwise vary with the problem |
| One numerical root near an initial guess | `nsolve` | Does not enumerate all roots or model measurement uncertainty |

Exact symbolic solving is most useful for small, exact models. A missing closed form, an empty result, and an inconsistent model are different situations and should not be conflated. Always validate returned candidates in the original equations.

## Practice: localize another source

**Timebox:** 10–15 minutes.

Three sensors are located at $(0,0)$, $(6,0)$, and $(0,8)$. Each reports an exact range of $5$.

1. Predict how many candidates the first two sensors determine, then solve for them.
2. Use the third measurement to identify the admissible candidate.
3. Subtract the first residual from the other two, solve the resulting linear system, and validate its result in all three original equations.
4. Replace the third range by $5.1=51/10$. Determine whether the exact system remains consistent and state what class of method should replace exact solving.

**Success criteria:** retain every candidate until a stated constraint rejects it, report residuals of the original equations, and distinguish an empty exact solution set from a failed computation.

**Hint:** reuse `range_equation` and `equation_residual`; request dictionaries from `solve` so that substitution is direct.

### Reference solution

In [20]:
practice_equation_a = range_equation((0, 0), 5)
practice_equation_b = range_equation((6, 0), 5)
practice_equation_c = range_equation((0, 8), 5)

practice_candidates = sp.solve(
    (practice_equation_a, practice_equation_b),
    (x, y),
    dict=True,
)
practice_candidate_checks = [
    (candidate, equation_residual(practice_equation_c, candidate))
    for candidate in practice_candidates
]

practice_candidates, practice_candidate_checks

In [21]:
practice_residuals = [
    equation.lhs - equation.rhs
    for equation in (
        practice_equation_a,
        practice_equation_b,
        practice_equation_c,
    )
]
practice_linearized = [
    sp.expand(practice_residuals[1] - practice_residuals[0]),
    sp.expand(practice_residuals[2] - practice_residuals[0]),
]
practice_linear_solution = sp.linsolve(practice_linearized, (x, y))
assert len(practice_linear_solution) == 1
practice_point = next(iter(practice_linear_solution))
practice_candidate = dict(zip((x, y), practice_point, strict=True))
practice_checks = tuple(
    equation_residual(equation, practice_candidate)
    for equation in (
        practice_equation_a,
        practice_equation_b,
        practice_equation_c,
    )
)

practice_linear_solution, practice_checks

In [22]:
practice_noisy_equation_c = range_equation(
    (0, 8),
    sp.Rational(51, 10),
)
practice_noisy_residual_c = (
    practice_noisy_equation_c.lhs
    - practice_noisy_equation_c.rhs
)
practice_noisy_solution = sp.nonlinsolve(
    (
        practice_residuals[0],
        practice_residuals[1],
        practice_noisy_residual_c,
    ),
    (x, y),
)

practice_noisy_solution

The first two sensors produce $(3,-4)$ and $(3,4)$. Only $(3,4)$ satisfies the third exact range, and subtracting residuals yields the same point with zero residual in every original equation. Changing the third range to $5.1$ makes the exact polynomial system inconsistent, so `nonlinsolve` returns the empty set. A numerical estimation method—normally weighted nonlinear least squares with an uncertainty model—is the appropriate continuation; choosing one of the old exact candidates or rounding until the equations agree would not be defensible.

## Conclusions and next steps

Equation solving is a reasoning process: define the domain, preserve all branches, incorporate independent constraints, and verify candidates in the original equations. Specialized solvers make solution structure clearer, but no API can decide whether an algebraic candidate is scientifically admissible.

The exact-to-inconsistent transition is equally important. An empty exact solution set can be the correct diagnosis for noisy measurements. Replacing that problem by an estimator requires an explicit objective and uncertainty assumptions.

For current API guidance, consult [SymPy's solving guide](https://docs.sympy.org/latest/guides/solving/). For numerical multidimensional root finding and residual-based acceptance, continue with [the multidimensional root-finding notebook](../numpy/multidimensional_root_finding.ipynb). The [pendulum introduction](pendulum_sympy_intro.ipynb) develops symbolic expressions, functions, calculus, limits, and series through a complementary running example.